In [5]:
import requests
from bs4 import BeautifulSoup 



In [6]:
shop_html = '''
<div id="catalog">
  <div class="item" data-pid="P101"><span class="title">게이밍 마우스</span><span class="cat">가전</span><span class="cost">45,000원</span></div>
  <div class="item" data-pid="P102"><span class="title">린넨 원피스</span><span class="cat"> 패션 </span><span class="cost"> 79,000 원 </span></div>
  <div class="item" data-pid="P103"><span class="title">아로마 캔들</span><span class="cat">뷰티</span><span class="cost">₩23,000</span></div>
  <div class="item" data-pid="P104"><span class="title">프리미엄 원두</span><span class="cat">식품</span><span class="cost"></span></div>
  <div class="item" data-pid="P105"><span class="title">요가 매트</span><span class="cat">스포츠</span><span class="cost">38,500원</span></div>
</div>
'''
shop_soup = BeautifulSoup(shop_html, "html.parser")
print("종합 실습 데이터 준비 완료 — 상품 카드:", len(shop_soup.select(".item")), "개")

종합 실습 데이터 준비 완료 — 상품 카드: 5 개


In [7]:
cards = shop_soup.select(".item")
print("[검증] 추출 대상 카드 수:", len(cards), "개")

rows = []
for c in cards:
    rows.append({
        "product_id": c.get("data-pid"),
        "name": c.select_one(".title").get_text(),
        "category": c.select_one(".cat").get_text(),
        "price": c.select_one(".cost").get_text(),
    })
print("[추출] 완료. 앞 2개(날것):")
for r in rows[:2]:
    print(" ", r)

[검증] 추출 대상 카드 수: 5 개
[추출] 완료. 앞 2개(날것):
  {'product_id': 'P101', 'name': '게이밍 마우스', 'category': '가전', 'price': '45,000원'}
  {'product_id': 'P102', 'name': '린넨 원피스', 'category': ' 패션 ', 'price': ' 79,000 원 '}


In [14]:
import pandas as pd
import numpy as np
from bs4 import BeautifulSoup

In [15]:
shop_df = pd.DataFrame(rows)

shop_df["price"] = (
    shop_df["price"].str.replace(r"[^0-9]", "", regex=True).replace("", np.nan)
)
shop_df["price"] = pd.to_numeric(shop_df["price"])

shop_df["category"] = shop_df["category"].str.strip()

print("[정제 결과]")
display(shop_df)
print("\n[결측 진단] 가격 결측:", int(shop_df["price"].isna().sum()), "건  (집계 시 제외 판단)")
print("[카테고리별 평균 가격]")
print(shop_df.dropna(subset=["price"]).groupby("category")["price"].mean().round(0))

[정제 결과]


,product_id,name,category,price
0,P101,게이밍 마우스,가전,45000.0
1,P102,린넨 원피스,패션,79000.0
2,P103,아로마 캔들,뷰티,23000.0
3,P104,프리미엄 원두,식품,NaN
4,P105,요가 매트,스포츠,38500.0



[결측 진단] 가격 결측: 1 건  (집계 시 제외 판단)
[카테고리별 평균 가격]
category
가전     45000.0
뷰티     23000.0
스포츠    38500.0
패션     79000.0
Name: price, dtype: float64


In [16]:
local_books_fallback = '''
<section>
  <article class="product_pod"><h3><a title="A Light in the Attic">A Light...</a></h3><p class="price_color">£51.77</p></article>
  <article class="product_pod"><h3><a title="Tipping the Velvet">Tipping...</a></h3><p class="price_color">£53.74</p></article>
  <article class="product_pod"><h3><a title="Soumission">Soumission</a></h3><p class="price_color">£50.10</p></article>
</section>
'''

books_url = "https://books.toscrape.com/"
try:
    if not HAS_REQUESTS:
        raise RuntimeError("requests 미설치")
    resp = requests.get(books_url, timeout=5, headers={"User-Agent": "modulabs-edu-bot/1.0"})
    resp.encoding = "utf-8"   # 사이트가 UTF-8이므로 인코딩을 명시 (£ 등 기호가 깨지는 것 방지)
    html = resp.text
    source = "실제 books.toscrape.com"
except Exception as e:
    html = local_books_fallback
    source = f"로컬 폴백({type(e).__name__})"

books_soup = BeautifulSoup(html, "html.parser")
# 같은 추출 패턴: 바깥에서 product_pod 모으고, 안에서 제목·가격 꺼내기
books = []
for pod in books_soup.select("article.product_pod"):
    books.append({
        "title": pod.h3.a.get("title"),
        "price": pod.select_one("p.price_color").get_text(strip=True),
    })

print("데이터 출처:", source)
print("[검증] 추출한 책 수:", len(books), "권")
display(pd.DataFrame(books).head())

데이터 출처: 로컬 폴백(NameError)
[검증] 추출한 책 수: 3 권


,title,price
0,A Light in the Attic,£51.77
1,Tipping the Velvet,£53.74
2,Soumission,£50.10


In [17]:
# 수집 결과 요약 숫자 자동 추출
n_items = len(shop_df)
n_priced = int(shop_df["price"].notna().sum())
n_missing = int(shop_df["price"].isna().sum())
avg_price = shop_df["price"].mean()
top_cat = shop_df.dropna(subset=["price"]).groupby("category")["price"].mean().idxmax()

print("자동 추출된 수집 리포트 숫자")
print(f"  수집 상품 수   : {n_items}개")
print(f"  가격 확보      : {n_priced}개 / 결측 {n_missing}개")
print(f"  평균 가격      : {avg_price:,.0f}원")
print(f"  최고가 카테고리: {top_cat}")

자동 추출된 수집 리포트 숫자
  수집 상품 수   : 5개
  가격 확보      : 4개 / 결측 1개
  평균 가격      : 46,375원
  최고가 카테고리: 패션


In [18]:
# 코드 퀴즈 — 모범 답안
quiz_html = '''
<div class="goods"><span class="gname">텀블러</span><span class="gprice">15,000원</span></div>
<div class="goods"><span class="gname">에코백</span><span class="gprice">9,900원</span></div>
'''
quiz_soup = BeautifulSoup(quiz_html, "html.parser")

result = []
for g in quiz_soup.select(".goods"):           # 바깥에서 카드 모으기
    result.append({
        "name": g.select_one(".gname").get_text(),    # 안에서 값 꺼내기
        "price": g.select_one(".gprice").get_text(),
    })
print("추출 결과:", result)
print("검증 — 추출 개수:", len(result), "개 (기대 2개)")

추출 결과: [{'name': '텀블러', 'price': '15,000원'}, {'name': '에코백', 'price': '9,900원'}]
검증 — 추출 개수: 2 개 (기대 2개)


# 📊 데이터 수집 리포트

## 1. 수집 개요

- **대상:** 모두마켓 상품 목록 페이지와 연습용 사이트인 books.toscrape.com
- **방법:** 내장 HTML 또는 requests로 데이터를 가져온 뒤, BeautifulSoup으로 상품 정보를 추출하고 DataFrame으로 변환하였다. 이후 문자열과 결측치를 정제하였다.
- **도구 선택 이유:** 상품 정보가 HTML 소스에 들어 있는 정적 페이지이므로, 브라우저 자동화 도구보다 빠르고 간단한 BeautifulSoup을 사용하였다.

## 2. 수집 결과

- 모두마켓 상품은 총 **5개** 수집하였다.
- 가격이 확보된 상품은 **4개**, 가격 결측 상품은 **1개**였다.
- 가격이 있는 상품의 평균 가격은 약 **46,375원**이었다.
- 평균 가격이 가장 높은 카테고리는 **패션**이었다.

## 3. 데이터 신뢰성

- 추출된 상품 카드가 총 5개인지 실제 HTML의 상품 수와 대조하여 확인하였다. 이를 통해 코드가 오류 없이 실행됐지만 결과가 잘못 나오는 조용한 실패를 방지하였다.
- 가격에 포함된 `원`, `₩`, 쉼표와 공백을 제거한 뒤 숫자형으로 변환하였다.
- 가격이 없는 상품 1개는 실제 가격을 알 수 없으므로 임의의 값으로 대체하지 않고, 평균 가격 등의 집계에서 제외하였다.
- 상품 카테고리의 앞뒤 공백도 제거하여 같은 카테고리가 서로 다른 값으로 처리되지 않도록 정제하였다.

## 4. 윤리 점검

- robots.txt를 확인하여 수집이 허용된 경로인지 점검하고, 연속 요청으로 사이트에 부담을 주지 않도록 요청 사이에 `time.sleep()`을 적용하였다.
- 실제 상업 사이트가 아닌 수집 연습이 허용된 샌드박스 사이트만 이용하였다.
- 상품명과 가격 등 공개된 상품 정보만 수집했으며 개인정보는 수집하지 않았다.